# 01 — Worker

**Run this unedited, every session, as many times as you like.** It claims
whatever the ledger says is next and runs it.

There is nothing to configure. The cell, scene and seed come from the ledger,
which enforces stage order and prerequisites globally — which is precisely why
this is one notebook rather than eight.

If a session dies mid-run, do nothing: the row is left with a stale heartbeat
and the next session reclaims it. Interrupted runs **restart** rather than
resume, deliberately — the checkpoint omits the medium model, the codebooks and
the loop's schedule flags, so resuming would silently reinitialise β and
produce a run that looks complete and is a different experiment.


## 1. Drive and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# The one place paths are defined. Everything else derives from DRIVE_ROOT.
#
#   e3dgsuw/
#     dataset/SeathruNeRF_dataset/   original, as downloaded
#     dataset/undistorted/<scene>/   PINHOLE + sparse/0/  <- required
#     dense/<scene>.ply|.json        M1 clouds, SHA-256 sidecars
#     run_ledger.json                campaign state
#     runs/<cell>/<scene>/s<seed>/   one run, all of it together
#     analysis/                      analyse.py output, figures, tables
# ---------------------------------------------------------------------------
DRIVE_ROOT   = '/content/drive/MyDrive/e3dgsuw'
DATA_ORIG    = f'{DRIVE_ROOT}/dataset/SeathruNeRF_dataset'
DATA_UNDIST  = f'{DRIVE_ROOT}/dataset/undistorted'
DENSE_DIR    = f'{DRIVE_ROOT}/dense'
ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

# Training reads from local disk, not Drive: the scene loader pulls every image
# at startup, and Drive's FUSE layer makes that far slower than a single copy.
LOCAL_DATA   = '/content/data'

REPO_URL  = 'https://github.com/dinanirham/An-Efficient-3D-Gaussian-Splatting-for-Underwater-3D-Reconstruction.git'
REPO_DIR  = '/content/e3dgsuw'
IMPL_DIR  = f'{REPO_DIR}/implementation'
SCENES    = ['Curasao', 'IUI3-RedSea', 'JapaneseGradens-RedSea', 'Panama']

import os
for d in (DRIVE_ROOT, DATA_UNDIST, DENSE_DIR, ANALYSIS_DIR):
    os.makedirs(d, exist_ok=True)
print('drive root:', DRIVE_ROOT)


## 2. GPU — must be an A100

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
cap  = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
print(f'torch {torch.__version__}  cuda {torch.version.cuda}  {name}  sm_{cap[0]}{cap[1]}')

# Every conclusion in this study is a between-cell contrast, and cells on
# different devices are not comparable. Stop now rather than produce a run
# that has to be discarded later.
assert 'A100' in name, f'Expected an A100, got {name!r}. Restart the runtime.'


## 3. Clone and build

In [ ]:
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git','clone','--depth','1',REPO_URL,REPO_DIR], check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'], check=True)
os.chdir(IMPL_DIR)
print(subprocess.run(['git','-C',REPO_DIR,'log','--oneline','-1'],
                     capture_output=True, text=True).stdout)

# Builds diff_gaussian_rasterization_ms and simple_knn for sm_80, and installs
# only the dependencies Colab does not already ship.
!bash tools/setup_colab.sh


## 4. Verify the rasterizer, then the dataset

In [ ]:
# The gate. The whole rasterizer merge rests on one identity: for a single
# Gaussian at depth z the probe gives Z_raw = alpha*z, so Z_raw/alpha must
# recover z on every covered pixel. Verified on sm_86 during development; this
# confirms it on sm_80 before anything is trained on top of it.
%cd $IMPL_DIR
!python -m tools.verify_rasterizer


In [ ]:
# Copy the undistorted scenes to local disk. The loader reads every image at
# startup; from Drive that is markedly slower than one bulk copy.
import os, shutil, time
os.makedirs(LOCAL_DATA, exist_ok=True)
t0 = time.time()
for s in SCENES:
    dst = f'{LOCAL_DATA}/{s}'
    if not os.path.exists(dst):
        shutil.copytree(f'{DATA_UNDIST}/{s}', dst)
print(f'dataset staged locally in {time.time()-t0:.0f}s')
!python -m tools.run_ledger status --output_root "$DRIVE_ROOT"


## 5. Work

`--max_minutes` should sit **below** the session limit so the loop stops
claiming new runs and exits cleanly rather than being killed mid-run.

If the budget has not been set yet, every M2 cell is blocked and the queue will
say so — that is expected until S1 (A0) completes.

In [ ]:
!python -m tools.run_queue \
    --output_root "$DRIVE_ROOT" \
    --data_root   "$LOCAL_DATA" \
    --max_minutes 200


## 6. After S1 completes — set the budget

The primitive budget comes from A0's converged count, which no publication of
the baseline reports. Until it is set, every M2 cell (A2, A4, A6, A7) stays
blocked.

A budget that does not bind makes A4 equivalent to A1 and A7 to A5, and a null
interaction measured in that state is a configuration artifact rather than a
finding — so check the counts before setting it.

In [ ]:
import glob, csv
counts = []
for f in sorted(glob.glob(f'{DRIVE_ROOT}/runs/A0/*/s*/diagnostics.csv')):
    rows = list(csv.DictReader(open(f)))
    if rows:
        counts.append((f.split('/runs/')[1], int(rows[-1]['n_primitives'])))
for name, n in counts:
    print(f'{n:>10,}  {name}')
if counts:
    import statistics
    print(f'\nmedian {statistics.median(n for _, n in counts):,.0f}')
    print('Set a budget BELOW these, so it binds:')
    print(f'  !python -m tools.run_ledger set-budget <count> --output_root "$DRIVE_ROOT"')


---
Re-run this notebook in a fresh session to continue. Nothing needs changing
between sessions.
